# P6 — Sensibilidad del umbral de coautoría

Ejecuta el pipeline temporal corregido de P4 para los umbrales **1, 2, 3 y 5**. Las características se calculan únicamente con el grafo histórico; el periodo futuro se utiliza solo para construir las etiquetas.

Los resultados se guardan en la subcarpeta `P6_sensibilidad_umbral`, por lo que no sobrescriben los archivos principales del pipeline. Si un umbral deja muy pocos positivos para separar train/validation, el programa no se detiene: guarda el conjunto de desarrollo y lo marca como insuficiente en el resumen.

In [1]:
from pathlib import Path
from itertools import combinations
from collections import Counter
import random
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.model_selection import train_test_split
import features as ft

# ============================================================
# P6 - Sensibilidad del umbral de coautoria
# Parte del pipeline temporal corregido. NO usa informacion futura
# para calcular features: el futuro se usa solo para etiquetas.
# ============================================================

BASE_DIR = Path('C:/Users/Dra. Pilar/UltimaversionProgramasComparacion')
OUTPUT_DIR = BASE_DIR / 'P6_sensibilidad_umbral'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 41
THRESHOLDS = [1, 2, 3, 5]
NEGATIVE_RATIO = 1
NEGATIVE_MODE = 'random'
VAL_SIZE = 0.20
MAX_POSITIVES_TRAINVAL = None
MAX_POSITIVES_TEST = None

random.seed(SEED)
np.random.seed(SEED)

FILES = {
    'full_2000_2020': BASE_DIR / 'BD_UNAM_Graph_2000_2020.gml',
    'sub_2000_2020':  BASE_DIR / 'BD_UNAM_Sub_Graph_2000_2020.gml',
    'full_2021_2022': BASE_DIR / 'BD_UNAM_Graph_2021_2022.gml',
    'sub_2021_2022':  BASE_DIR / 'BD_UNAM_Sub_Graph_2021_2022.gml',
    'full_2023_2024': BASE_DIR / 'BD_UNAM_Graph_2023_2024.gml',
    'sub_2023_2024':  BASE_DIR / 'BD_UNAM_Sub_Graph_2023_2024.gml',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos GML:\n' + '\n'.join(missing))

G_2000_2020 = nx.read_gml(FILES['full_2000_2020'])
S_2000_2020 = nx.read_gml(FILES['sub_2000_2020'])
G_2021_2022 = nx.read_gml(FILES['full_2021_2022'])
S_2021_2022 = nx.read_gml(FILES['sub_2021_2022'])
G_2023_2024 = nx.read_gml(FILES['full_2023_2024'])
S_2023_2024 = nx.read_gml(FILES['sub_2023_2024'])

G_HIST_2022 = nx.compose(G_2000_2020, G_2021_2022)
S_HIST_2022 = nx.compose(S_2000_2020, S_2021_2022)

print('Grafos cargados:')
for name, G in [
    ('G_2000_2020', G_2000_2020),
    ('G_2021_2022', G_2021_2022),
    ('G_2023_2024', G_2023_2024),
    ('G_HIST_2022', G_HIST_2022),
]:
    print(f'{name}: nodos={G.number_of_nodes():,}, aristas={G.number_of_edges():,}')




C:\Users\pilarang\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Grafos cargados:
G_2000_2020: nodos=9,041, aristas=14,064
G_2021_2022: nodos=5,151, aristas=6,752
G_2023_2024: nodos=3,207, aristas=4,025
G_HIST_2022: nodos=13,371, aristas=20,728


## Funciones auxiliares y construcción temporal de pares

In [2]:
def node_type(G, node):
    return str(G.nodes[node].get('tipo', '')).strip().lower()


def is_article_node(G, node):
    return node_type(G, node) == 'articulo'


def is_author_node(G, node):
    return node_type(G, node) == 'autor'


def author_nodes(G):
    return {n for n in G.nodes if is_author_node(G, n)}


def add_isolated_historical_authors(S_history, G_history):
    S = S_history.copy()
    hist_authors = author_nodes(G_history)
    missing_authors = hist_authors - set(S.nodes())
    S.add_nodes_from(missing_authors)
    return S, missing_authors


def canon_pair(u, v):
    return tuple(sorted((str(u), str(v))))


def coauthorship_counts(G, eligible_authors=None):
    eligible = None if eligible_authors is None else set(eligible_authors)
    counts = Counter()
    for article in G.nodes:
        if not is_article_node(G, article):
            continue
        authors = [
            a for a in G.neighbors(article)
            if is_author_node(G, a) and (eligible is None or a in eligible)
        ]
        authors = sorted(set(authors), key=str)
        for u, v in combinations(authors, 2):
            counts[canon_pair(u, v)] += 1
    return counts


def threshold_edges(G, threshold, eligible_authors=None):
    counts = coauthorship_counts(G, eligible_authors)
    return {pair for pair, n in counts.items() if n >= threshold}


def safe_feature(func, graph, u, v, default=np.nan):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else value
    except Exception:
        return default


def safe_len_feature(func, graph, u, v, default=0):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else len(value)
    except Exception:
        return default


def paper_count(G, author):
    if author not in G:
        return 0
    return sum(1 for n in G.neighbors(author) if is_article_node(G, n))


def shortest_path_safe(G, u, v):
    try:
        if u not in G or v not in G:
            return np.nan
        return nx.shortest_path_length(G, u, v)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return np.nan


def historical_hardness(G_sub, u, v):
    if u not in G_sub or v not in G_sub:
        return False
    try:
        if len(list(nx.common_neighbors(G_sub, u, v))) > 0:
            return True
    except Exception:
        pass
    try:
        return nx.shortest_path_length(G_sub, u, v) <= 3
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return False


def build_forecast_pairs(
    G_history, G_future, S_history, threshold=1, negative_ratio=1,
    negative_mode='random', max_positives=None, seed=41
):
    rng = random.Random(seed)
    hist_authors = author_nodes(G_history)
    future_authors = author_nodes(G_future)
    structural_authors = set(S_history.nodes())

    eligible = sorted(hist_authors.intersection(future_authors), key=str)
    if len(eligible) < 2:
        raise ValueError('No hay suficientes autores elegibles para construir pares.')

    if any(not is_author_node(G_history, n) for n in eligible):
        raise AssertionError('La cohorte contiene nodos que no son autores en historia.')
    if any(not is_author_node(G_future, n) for n in eligible):
        raise AssertionError('La cohorte contiene nodos que no son autores en futuro.')

    hist_positive = threshold_edges(G_history, threshold, eligible_authors=eligible)
    future_positive_all = threshold_edges(G_future, threshold, eligible_authors=eligible)
    future_positive = sorted(future_positive_all - hist_positive, key=lambda x: (x[0], x[1]))

    if max_positives is not None and len(future_positive) > max_positives:
        future_positive = rng.sample(future_positive, max_positives)

    positives = set(future_positive)
    n_neg = len(positives) * int(negative_ratio)
    if n_neg == 0:
        raise ValueError('No se encontraron positivos nuevos con el umbral elegido.')

    negatives = set()
    attempts = 0
    max_attempts = max(100000, n_neg * 500)
    while len(negatives) < n_neg and attempts < max_attempts:
        attempts += 1
        u, v = rng.sample(eligible, 2)
        pair = canon_pair(u, v)
        if pair in hist_positive or pair in positives or pair in negatives:
            continue
        if negative_mode == 'hard' and not historical_hardness(S_history, u, v):
            continue
        negatives.add(pair)

    if len(negatives) < n_neg:
        raise RuntimeError(
            f'No fue posible obtener {n_neg} negativos {negative_mode}; '
            f'se obtuvieron {len(negatives)}.'
        )

    rows = [(u, v, 1) for u, v in positives] + [(u, v, 0) for u, v in negatives]
    df = pd.DataFrame(rows, columns=['source', 'target', 'connected'])
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    audit = {
        'historical_authors': len(hist_authors),
        'future_authors': len(future_authors),
        'historical_structural_nodes': len(structural_authors),
        'historical_structural_active_authors': sum(
            1 for n in S_history.nodes if S_history.degree(n) > 0
        ),
        'eligible_authors': len(eligible),
        'eligible_with_structural_links': sum(
            1 for n in eligible if n in S_history and S_history.degree(n) > 0
        ),
        'eligible_structural_isolates': sum(
            1 for n in eligible if n in S_history and S_history.degree(n) == 0
        ),
        'historical_positive_pairs': len(hist_positive),
        'future_positive_pairs_all': len(future_positive_all),
        'new_future_positive_pairs': len(positives),
        'negative_pairs': len(negatives),
        'negative_ratio': negative_ratio,
        'negative_mode': negative_mode,
        'threshold': threshold,
    }
    return df, audit


def add_historical_features(df, G_history, S_history):
    out = df.copy()

    missing_structural = (set(out['source']) | set(out['target'])) - set(S_history.nodes())
    if missing_structural:
        raise AssertionError(
            f'Faltan {len(missing_structural)} autores historicos en S_history.'
        )

    # FP
    out['sum_of_papers'] = out.apply(
        lambda r: paper_count(G_history, r['source']) + paper_count(G_history, r['target']),
        axis=1,
    )
    out['sum_of_keywords'] = out.apply(
        lambda r: safe_len_feature(ft.sum_keywords, G_history, r['source'], r['target'], 0), axis=1
    )
    out['keywords_match'] = out.apply(
        lambda r: safe_len_feature(ft.keyword_match, G_history, r['source'], r['target'], 0), axis=1
    )
    out['sum_of_areas'] = out.apply(
        lambda r: safe_len_feature(ft.sum_areas, G_history, r['source'], r['target'], 0), axis=1
    )

    # LSP
    out['sum_of_neighbors'] = out.apply(
        lambda r: len(ft.neighbors(G_history, r['source'])) + len(ft.neighbors(G_history, r['target']))
        if r['source'] in G_history and r['target'] in G_history else 0,
        axis=1,
    )
    out['log_secundary_neighbors'] = out.apply(
        lambda r: safe_feature(ft.sum_sec_neig, G_history, r['source'], r['target'], 0.0), axis=1
    )
    out['lenght_short_path'] = out.apply(
        lambda r: shortest_path_safe(S_history, r['source'], r['target']), axis=1
    )
    out['clustering_index_sum'] = out.apply(
        lambda r: safe_feature(ft.clustering_index, S_history, r['source'], r['target'], np.nan), axis=1
    )
    out['jaccard_coefficient'] = out.apply(
        lambda r: safe_feature(ft.jaccard_coefficient, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['resource_allocation'] = out.apply(
        lambda r: safe_feature(ft.get_resource_allocation_index, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['adamic_adar_index'] = out.apply(
        lambda r: safe_feature(ft.get_adamic_adar_index, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['preferential_attachment'] = out.apply(
        lambda r: safe_feature(ft.get_preferential_attachment, S_history, r['source'], r['target'], 0.0), axis=1
    )

    # GSP
    out['community_cn'] = out.apply(
        lambda r: safe_feature(ft.community_cn, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['community_ra'] = out.apply(
        lambda r: safe_feature(ft.community_ra, S_history, r['source'], r['target'], 0.0), axis=1
    )
    return out


def summarize_df(df):
    counts = df['connected'].value_counts().sort_index().to_dict()
    return {
        'n': len(df),
        'n_negative': int(counts.get(0, 0)),
        'n_positive': int(counts.get(1, 0)),
        'positive_rate': float(df['connected'].mean()) if len(df) else np.nan,
        'duplicates': int(df.duplicated(['source', 'target']).sum()) if len(df) else 0,
    }




## Ejecución de los cuatro umbrales y generación del resumen

In [3]:
# Incorporar autores historicos sin coautorias como nodos aislados.
S_2000_2020, ISOLATED_2000_2020 = add_isolated_historical_authors(
    S_2000_2020, G_2000_2020
)
S_HIST_2022, ISOLATED_2000_2022 = add_isolated_historical_authors(
    S_HIST_2022, G_HIST_2022
)

print(f'Autores historicos anadidos como aislados 2000-2020: {len(ISOLATED_2000_2020):,}')
print(f'Autores historicos anadidos como aislados 2000-2022: {len(ISOLATED_2000_2022):,}')

summary_rows = []
audit_rows = []

for threshold in THRESHOLDS:
    print('\n' + '=' * 72)
    print(f'THRESHOLD = {threshold}')
    print('=' * 72)

    suffix = f'thr{threshold}_neg{NEGATIVE_RATIO}_{NEGATIVE_MODE}_isolates'
    row = {
        'threshold': threshold,
        'negative_ratio': NEGATIVE_RATIO,
        'negative_mode': NEGATIVE_MODE,
        'status': 'pending',
        'train_file_written': False,
        'val_file_written': False,
        'test_file_written': False,
    }

    try:
        # Desarrollo: historia 2000-2020 -> futuro 2021-2022
        development, audit_dev = build_forecast_pairs(
            G_2000_2020,
            G_2021_2022,
            S_2000_2020,
            threshold=threshold,
            negative_ratio=NEGATIVE_RATIO,
            negative_mode=NEGATIVE_MODE,
            max_positives=MAX_POSITIVES_TRAINVAL,
            seed=SEED,
        )
        audit_rows.append({
            'threshold': threshold,
            'split': 'development',
            'feature_history': '2000-2020',
            'label_horizon': '2021-2022',
            **audit_dev,
        })

        dev_counts = development['connected'].value_counts()
        row.update({
            'development_n': len(development),
            'development_positive': int(dev_counts.get(1, 0)),
            'development_negative': int(dev_counts.get(0, 0)),
        })

        # Guardar development SIEMPRE, incluso si no alcanza para stratify.
        development_raw_file = OUTPUT_DIR / (
            f'sample_temporal_development_hist2000_2020_target2021_2022_{suffix}.csv'
        )

        # Features historicas solo si hay pares.
        development = add_historical_features(development, G_2000_2020, S_2000_2020)
        development.to_csv(development_raw_file, index=False)
        row['development_file'] = str(development_raw_file)

        if len(dev_counts) < 2 or dev_counts.min() < 2:
            row['status'] = 'insufficient_development_for_stratified_train_val'
            row['note'] = (
                'Se guardo development, pero no se generaron train/val porque '
                'alguna clase tiene menos de 2 observaciones.'
            )
        else:
            sample_train, sample_val = train_test_split(
                development,
                test_size=VAL_SIZE,
                random_state=SEED,
                stratify=development['connected'],
            )
            sample_train = sample_train.reset_index(drop=True)
            sample_val = sample_val.reset_index(drop=True)

            train_file = OUTPUT_DIR / (
                f'sample_temporal_train_hist2000_2020_target2021_2022_{suffix}.csv'
            )
            val_file = OUTPUT_DIR / (
                f'sample_temporal_val_hist2000_2020_target2021_2022_{suffix}.csv'
            )
            sample_train.to_csv(train_file, index=False)
            sample_val.to_csv(val_file, index=False)

            row.update({
                'train_file_written': True,
                'val_file_written': True,
                'train_file': str(train_file),
                'val_file': str(val_file),
                **{f'train_{k}': v for k, v in summarize_df(sample_train).items()},
                **{f'val_{k}': v for k, v in summarize_df(sample_val).items()},
            })

        # Test: historia 2000-2022 -> futuro 2023-2024
        sample_test, audit_test = build_forecast_pairs(
            G_HIST_2022,
            G_2023_2024,
            S_HIST_2022,
            threshold=threshold,
            negative_ratio=NEGATIVE_RATIO,
            negative_mode=NEGATIVE_MODE,
            max_positives=MAX_POSITIVES_TEST,
            seed=SEED + 1,
        )
        audit_rows.append({
            'threshold': threshold,
            'split': 'test',
            'feature_history': '2000-2022',
            'label_horizon': '2023-2024',
            **audit_test,
        })

        sample_test = add_historical_features(sample_test, G_HIST_2022, S_HIST_2022)
        test_file = OUTPUT_DIR / (
            f'sample_temporal_test_hist2000_2022_target2023_2024_{suffix}.csv'
        )
        sample_test.to_csv(test_file, index=False)
        row.update({
            'test_file_written': True,
            'test_file': str(test_file),
            **{f'test_{k}': v for k, v in summarize_df(sample_test).items()},
        })

        if row['status'] == 'pending':
            row['status'] = 'ok'

    except Exception as exc:
        row['status'] = 'error'
        row['note'] = f'{type(exc).__name__}: {exc}'
        print(f'ERROR threshold={threshold}: {row["note"]}')

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
audit_df = pd.DataFrame(audit_rows)

summary_file = OUTPUT_DIR / 'P6_threshold_sensitivity_summary.csv'
audit_file = OUTPUT_DIR / 'P6_threshold_sensitivity_audit.csv'
summary_df.to_csv(summary_file, index=False)
audit_df.to_csv(audit_file, index=False)

print('\n' + '=' * 72)
print('RESUMEN DE SENSIBILIDAD')
print('=' * 72)
cols = [
    'threshold', 'status',
    'development_n', 'development_positive', 'development_negative',
    'train_n', 'val_n', 'test_n', 'test_positive', 'test_negative'
]
print(summary_df[[c for c in cols if c in summary_df.columns]].to_string(index=False))
print('\nArchivos resumen:')
print(summary_file)
print(audit_file)
print('\nLos CSV individuales se guardaron en:')
print(OUTPUT_DIR)


Autores historicos anadidos como aislados 2000-2020: 0
Autores historicos anadidos como aislados 2000-2022: 0

THRESHOLD = 1

THRESHOLD = 2
ERROR threshold=2: ValueError: The test_size = 1 should be greater or equal to the number of classes = 2

THRESHOLD = 3
ERROR threshold=3: ValueError: No se encontraron positivos nuevos con el umbral elegido.

THRESHOLD = 5
ERROR threshold=5: ValueError: No se encontraron positivos nuevos con el umbral elegido.

RESUMEN DE SENSIBILIDAD
 threshold status  development_n  development_positive  development_negative  train_n  val_n  test_n
         1     ok           34.0                  17.0                  17.0     27.0    7.0   112.0
         2  error            4.0                   2.0                   2.0      NaN    NaN     NaN
         3  error            2.0                   1.0                   1.0      NaN    NaN     NaN
         5  error            NaN                   NaN                   NaN      NaN    NaN     NaN

Archivos resumen